# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://www.womenonwings.com")
links

['#content',
 'https://www.womenonwings.com/',
 '#',
 'https://www.womenonwings.com/',
 'https://www.womenonwings.com/impact/',
 'https://www.womenonwings.com/impact/',
 'https://www.womenonwings.com/case-studies/',
 'https://www.womenonwings.com/category/get-inspired/',
 'https://www.womenonwings.com/our-services/',
 'https://www.womenonwings.com/our-services/',
 'https://www.womenonwings.com/social-enterprises/',
 'https://www.womenonwings.com/government-institutions/',
 'https://www.womenonwings.com/about-us/',
 'https://www.womenonwings.com/about-us/',
 'https://www.womenonwings.com/team/',
 'https://www.womenonwings.com/experts/',
 'https://www.womenonwings.com/funders/',
 'https://www.womenonwings.com/network-partners/',
 'https://www.womenonwings.com/category/news/',
 'https://www.womenonwings.com/governance/',
 'https://www.womenonwings.com/annual-reports/',
 'https://www.womenonwings.com/how-we-came-about/',
 'https://www.womenonwings.com/contact/',
 'https://www.womenonwings.

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links. Include only links that link inside the domain of the website.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://www.womenonwings.com"))


Here is the list of links on the website https://www.womenonwings.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#content
https://www.womenonwings.com/
#
https://www.womenonwings.com/
https://www.womenonwings.com/impact/
https://www.womenonwings.com/impact/
https://www.womenonwings.com/case-studies/
https://www.womenonwings.com/category/get-inspired/
https://www.womenonwings.com/our-services/
https://www.womenonwings.com/our-services/
https://www.womenonwings.com/social-enterprises/
https://www.womenonwings.com/government-institutions/
https://www.womenonwings.com/about-us/
https://www.womenonwings.com/about-us/
https://www.womenonwings.com/team/
https://www.womenonwings.com/experts/
https://www.womenonwings.com/funders/
https://www.womenonwings.com/network-partners/
https://www.womenonwings.com/cat

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://www.womenonwings.com")

{'links': [{'type': 'about page',
   'url': 'https://www.womenonwings.com/about-us/'},
  {'type': 'history page',
   'url': 'https://www.womenonwings.com/how-we-came-about/'},
  {'type': 'services page',
   'url': 'https://www.womenonwings.com/our-services/'},
  {'type': 'offer page', 'url': 'https://www.womenonwings.com/our-offer/'},
  {'type': 'case studies',
   'url': 'https://www.womenonwings.com/case-studies/'},
  {'type': 'impact', 'url': 'https://www.womenonwings.com/impact/'},
  {'type': 'social enterprises',
   'url': 'https://www.womenonwings.com/social-enterprises/'},
  {'type': 'government institutions',
   'url': 'https://www.womenonwings.com/government-institutions/'},
  {'type': 'team', 'url': 'https://www.womenonwings.com/team/'},
  {'type': 'experts', 'url': 'https://www.womenonwings.com/experts/'},
  {'type': 'funders', 'url': 'https://www.womenonwings.com/funders/'},
  {'type': 'network partners',
   'url': 'https://www.womenonwings.com/network-partners/'},
  {'type'

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://viaindia.nl")

Selecting relevant links for https://viaindia.nl by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://viaindia.nl/'},
  {'type': 'blog page', 'url': 'https://viaindia.nl/blogs/news'},
  {'type': 'blog post',
   'url': 'https://viaindia.nl/blogs/news/nakshi-kantha-borduursters'},
  {'type': 'blog post',
   'url': 'https://viaindia.nl/blogs/news/kantha-de-draad-van-leven'},
  {'type': 'collection page', 'url': 'https://viaindia.nl/collections/kantha'},
  {'type': 'collection page',
   'url': 'https://viaindia.nl/collections/nakshi-kantha'},
  {'type': 'collection page',
   'url': 'https://viaindia.nl/collections/alle-sjaals'},
  {'type': 'collection page',
   'url': 'https://viaindia.nl/collections/sjaals-biologisch-katoen'},
  {'type': 'collection page',
   'url': 'https://viaindia.nl/collections/dameskleding'},
  {'type': 'collection page',
   'url': 'https://viaindia.nl/collections/stoffen'},
  {'type': 'collection page',
   'url': 'https://viaindia.nl/collections/sieraden'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://www.womenonwings.com"))

Selecting relevant links for https://www.womenonwings.com by calling gpt-5-nano
Found 19 relevant links
## Landing Page:

Consulting for social entrepreneurship - Women on Wings

Skip to content
Menu
Menu
Impact
Impact
Case studies
Success stories
Our services
Our services
Social enterprise partners
Government institution partners
About us
About us
Meet the team
Experts
Funders
Network partners
News
Governance
Annual reports and accounts
How we came about
Contact
Donate
Menu
Impact
Impact
Case studies
Success stories
Our services
Our services
Social enterprise partners
Government institution partners
About us
About us
Meet the team
Experts
Funders
Network partners
News
Governance
Annual reports and accounts
How we came about
Contact
Donate
Co-create one million jobs for women in rural India
Indian social enterprises and government institutions receive our tailor-made pro bono consulting so they can scale and employ more women.
Manifesto
Meaningful work that brings confidence and self-r

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("Women on Wings", "https://www.womenonwings.com")

Selecting relevant links for https://www.womenonwings.com by calling gpt-5-nano
Found 27 relevant links


KeyboardInterrupt: 

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Women on Wings", "https://www.womenonwings.com")

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("Women on Wings", "https://www.womenonwings.com")

Selecting relevant links for https://www.womenonwings.com by calling gpt-5-nano
Found 17 relevant links


# Women on Wings Brochure

---

## Who We Are

**Women on Wings** is a social entrepreneurship consulting organization dedicated to co-creating economic opportunities for women in rural India. Since 2007, we have been providing tailor-made, pro bono business consultancy and mentoring to Indian social enterprises and government institutions. Our mission is to empower women by helping these organizations grow and scale, thereby creating meaningful jobs and sustainable income for women in rural areas.

---

## Our Mission & Vision

- **Mission:** Co-create one million jobs for women in rural India through economic empowerment.
- **Vision:** A future where every woman in rural India realizes her full potential, gains confidence, self-respect, and economic independence.
- **Manifesto:** Meaningful work that supports families, brings self-worth, and fosters sustainable futures.

---

## What We Do

We provide expert, customized business consultancy delivered by a powerful network of over 60 experienced Indian and Dutch business professionals. Our support covers key business areas including:

- Strategy & Planning
- Marketing & Sales
- Supply Chain Management
- Finance & Operations

This practical on-the-ground advice helps social enterprises and government bodies to expand their reach and impact, directly creating more jobs and sustainable livelihoods for women.

---

## Our Impact

- Over **456,000 jobs** co-created for women in rural India since 2007.
- Collaboration with **66 social enterprise partners** and numerous government institutions.
- Proven contribution to the economic and social development of rural communities.
- A unique pro bono consultancy model blending business expertise with social commitment.

---

## Our Customers & Partners

- **Indian Social Enterprises:** Empowering grassroots organizations to scale business models focused on women’s employment.
- **Government Institution Partners:** Assisting government-led initiatives aimed at women’s economic development.
- Supported by a network of enthusiastic partners, funders, and experts from both India and the Netherlands.

---

## Company Culture

Women on Wings thrives on a culture of meaningful social impact, collaboration, and professional excellence. Our team and volunteers share a deep commitment to gender equality and sustainable development. We embrace diversity, learning, and the power of collective effort to create lasting change.

---

## Careers & Volunteering

At Women on Wings, we welcome professionals passionate about social entrepreneurship and women’s empowerment. We offer opportunities to:

- Join as an expert consultant providing professional advice pro bono.
- Work within a vibrant network of like-minded professionals from diverse business backgrounds.
- Engage in meaningful projects that produce sustainable, measurable impact.
  
Whether you are an experienced business leader, marketing specialist, supply chain expert, or finance professional, your skills can help transform millions of lives.

---

## Get Involved

- Partner with us to scale social enterprises and create jobs.
- Support us through funding or donations to further extend our impact.
- Join our network of experts or volunteer your professional skills.

---

## Contact

For more information, partnership inquiries, or to join our expert network, please visit our website or contact us directly.

---

*Empowering women, strengthening communities. Together, creating a better future for rural India.*  
**Women on Wings** | www.womenonwings.org

In [ ]:
stream_brochure("Via India", "https://viaindia.nl")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>